# Baseline en test con LM Studio

Evalúa el modelo sin entrenar sobre `datos/test.jsonl`: 20 árboles por nivel en los niveles 2 a 6, con 3 intentos por árbol (300 episodios).

**Si cambias un presupuesto:** cambia también `NOMBRE` y `RUTA_RESULTADOS`, para no mezclar corridas.

**Antes de empezar:** abre LM Studio, carga el modelo e inicia el servidor local. Abre este notebook desde la carpeta `seleccion_arbol` y selecciona el kernel del entorno de conda `tipico`, que tiene `pandas` y `numpy`.

**Si se interrumpe:** ejecuta todo de nuevo. Cada episodio se guarda al terminar, así que continúa donde quedó. Si cambias de modelo, cambia también `NOMBRE`, o sus resultados se mezclarían en el mismo archivo con ese nombre.

## Configuración

In [1]:
from modelo import ModeloLMStudio

NOMBRE = 'Qwen 3.5 2B'
MODELO_LM_STUDIO = 'qwen3.5-2b'
URL = 'http://localhost:1235/v1'
RUTA_DATOS = 'datos/test.jsonl'
RUTA_RESULTADOS = 'resultados/Qwen 3.5 2B/baseline_test.jsonl'
NIVELES = [2, 3, 4, 5, 6]
INTENTOS = 3  # El k de pass@k
TOKENS_PENSAMIENTO = 3400  # Al agotarlos, se cierra el razonamiento y el modelo debe responder
TOKENS_RESPUESTA = 200  # Tokens para escribir los nodos después de ese cierre

modelo = ModeloLMStudio(
    nombre=MODELO_LM_STUDIO,
    url_base=URL,
    tokens_pensamiento=TOKENS_PENSAMIENTO,
    tokens_respuesta=TOKENS_RESPUESTA,
)

## Comprobar la conexión

Si esta celda falla, revisa que el servidor de LM Studio esté iniciado.

In [2]:
import json
import urllib.request

with urllib.request.urlopen(f'{URL}/models', timeout=10) as respuesta:
    disponibles = [m['id'] for m in json.load(respuesta)['data']]

print('Modelos disponibles:', disponibles)
assert MODELO_LM_STUDIO in disponibles, f'{MODELO_LM_STUDIO} no está disponible en LM Studio'

Modelos disponibles: ['qwen3.5-2b', 'text-embedding-nomic-embed-text-v1.5']


## Evaluación

In [3]:
from evaluacion import evaluar_dataset

evaluar_dataset(modelo, NOMBRE, RUTA_DATOS, RUTA_RESULTADOS, niveles=NIVELES, intentos=INTENTOS)

Qwen 2B sin entrenar nivel 2 ep test-2-9 intento 2: recompensa=0.438 puntaje=7 óptimo=10 112.8s 3404tok
Qwen 2B sin entrenar nivel 2 ep test-2-9 intento 3: recompensa=1.000 puntaje=10 óptimo=10 91.5s 3406tok
Qwen 2B sin entrenar nivel 2 ep test-2-10 intento 1: recompensa=0.250 puntaje=3 óptimo=8 97.1s 3406tok
Qwen 2B sin entrenar nivel 2 ep test-2-10 intento 2: recompensa=0.500 puntaje=6 óptimo=8 89.4s 3404tok
Qwen 2B sin entrenar nivel 2 ep test-2-10 intento 3: recompensa=0.250 puntaje=3 óptimo=8 90.4s 3406tok
Qwen 2B sin entrenar nivel 2 ep test-2-6 intento 1: recompensa=1.000 puntaje=9 óptimo=9 123.6s 3406tok
Qwen 2B sin entrenar nivel 2 ep test-2-6 intento 2: recompensa=1.000 puntaje=9 óptimo=9 96.2s 3406tok
Qwen 2B sin entrenar nivel 2 ep test-2-6 intento 3: recompensa=0.889 puntaje=8 óptimo=9 93.5s 3404tok
Qwen 2B sin entrenar nivel 2 ep test-2-8 intento 1: recompensa=0.250 puntaje=4 óptimo=16 90.2s 3406tok
Qwen 2B sin entrenar nivel 2 ep test-2-8 intento 2: recompensa=1.000 punt

## Baseline por nivel

Puedes ejecutar estas celdas en cualquier momento para ver lo avanzado. Si la evaluación corre en otro proceso, no vuelvas a ejecutar la celda de evaluación: se repetirían episodios.

Las columnas de proporción van de 0 a 1.

- **pass@1:** intentos que logran el óptimo.
- **pass@3:** árboles donde al menos uno de sus 3 intentos logra el óptimo.
- **recompensa:** media del episodio. Igualar el mejor atajo vale 0,5, y una respuesta inválida, -1.
- **optimos:** árboles donde el modelo logró el mejor puntaje posible.
- **optimos_atajo:** óptimos de la estrategia "sin vecinos", que no razona sobre costos. El modelo debería superarla para mostrar que optimiza de verdad.
- **supera_voraz:** veces que superó la regla simple.
- **supera_atajos:** veces que superó al mejor atajo.
- **invalidos:** respuestas sin un formato válido.
- **cortes:** episodios que agotaron el pensamiento, y cuya respuesta fue forzada.

In [4]:
from evaluacion import resumir, resumir_pass

resumir_pass(RUTA_RESULTADOS, k=INTENTOS)

intentos  pass@1  arboles_completos  pass@3
modelo               nivel                                             
Qwen 2B sin entrenar 2            60   0.150                 20    0.25
                     3            60   0.033                 20    0.10
                     4            60   0.000                 20    0.00
                     5            60   0.000                 20    0.00
                     6            60   0.000                 20    0.00

In [5]:
resumir(RUTA_RESULTADOS)

episodios  recompensa  optimos  optimos_atajo  \
modelo               nivel                                                  
Qwen 2B sin entrenar 2             60       0.164    0.150            0.7   
                     3             60       0.222    0.033            0.0   
                     4             60       0.003    0.000            0.0   
                     5             60       0.104    0.000            0.0   
                     6             60       0.095    0.000            0.0   

                            supera_voraz  supera_atajos  invalidos  cortes  \
modelo               nivel                                                   
Qwen 2B sin entrenar 2             0.500          0.100      0.233   1.000   
                     3             0.483          0.100      0.083   0.983   
                     4             0.250          0.017      0.167   1.000   
                     5             0.350          0.033      0.083   1.000   
                     6             0.200          0.000      0.033   1.000   

                            segundos    tokens  
modelo               nivel                      
Qwen 2B sin entrenar 2        93.585  3405.917  
                     3        89.872  3361.683  
                     4        97.705  3407.000  
                     5        93.572  3409.633  
                     6        94.323  3408.217

## Baseline total

In [6]:
resumir_pass(RUTA_RESULTADOS, k=INTENTOS, por=['modelo'])

,intentos,pass@1,arboles_completos,pass@3
modelo,,,,
Qwen 2B sin entrenar,300,0.037,100,0.07


In [7]:
resumir(RUTA_RESULTADOS, por=['modelo'])

,episodios,recompensa,optimos,optimos_atajo,supera_voraz,supera_atajos,invalidos,cortes,segundos,tokens
modelo,,,,,,,,,,
Qwen 2B sin entrenar,300,0.117,0.037,0.14,0.357,0.05,0.12,0.997,93.811,3398.49
